# Module 17: K-Nearest Neighbour (KNN)  

This notebook covers the following sections for **KNN Classification**:
1. **Training & Prediction Process**
2. **Implementing KNN on a Dataset**
3. **Model Evaluation & Optimization**


## 0) Setup
We will use:
- `StandardScaler` because KNN depends on distances
- `KNeighborsClassifier` for classification
- A clean sklearn `Pipeline` to learn the correct workflow


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report
)

np.random.seed(42)


# 1) Training & Prediction Process (sklearn view)

### What “training” means for KNN
KNN is called a **lazy learner** because it does not learn weights or coefficients.

In practice, `.fit()` does two important things:
- Fits the scaler on training data (learns mean and standard deviation)
- Stores the training examples inside the KNN model

So training is fast, but prediction can be expensive.

### What prediction means
During `.predict()` KNN does:
1. Compute distance from the query point to all stored training points  
2. Sort distances  
3. Select the K nearest neighbors  
4. Majority vote for the predicted class  

That is why KNN can become slow when the dataset is large.


## Why scaling is non-negotiable
KNN uses distance. Distance is scale-sensitive.

If one feature has a much larger range than others, it dominates distance and breaks neighbor selection.
So we always place `StandardScaler` before KNN in the pipeline.


# 2) Implementing KNN on a Dataset

We use the **Wine dataset** (3 classes). It is good for learning because:
- All features are numeric
- Multi-class classification feels realistic
- Scaling matters

Pipeline we follow:
1. Train-test split  
2. Pipeline = scaling + KNN  
3. Fit on training data  
4. Predict on test data  
5. Evaluate  


In [2]:
wine = load_wine()
X = wine.data
y = wine.target

print("Dataset shape:", X.shape)
print("Classes:", list(wine.target_names))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)


Dataset shape: (178, 13)
Classes: [np.str_('class_0'), np.str_('class_1'), np.str_('class_2')]


In [3]:
# Baseline KNN model (Euclidean distance via Minkowski p=2)
clf = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=15, metric="minkowski", p=2))
])

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

In [4]:
print("Accuracy:", float(accuracy_score(y_test, y_pred)))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=wine.target_names))

Accuracy: 1.0

Confusion Matrix:
 [[15  0  0]
 [ 0 18  0]
 [ 0  0 12]]

Classification Report:
               precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        15
     class_1       1.00      1.00      1.00        18
     class_2       1.00      1.00      1.00        12

    accuracy                           1.00        45
   macro avg       1.00      1.00      1.00        45
weighted avg       1.00      1.00      1.00        45

